# 05 — Segmentation Training

Trains segmentation models over the design axes. Driven by
`stages/train_segmentation.py :: main()`.

**Scenarios** (`exps`): `single_date`, `mt_ndvi`, `gsi`, `rf`  
**Architectures** (`archs`): `deeplabv3plus_cbam` (ResNet-50), `segformer` (MiT-B2)  
**Normalization** (`norm_mode`): `percentile` *(main)*, `minmax`, `zscore`  
**Loss** (`loss`): `dynamic_balanced` = **DECB-CE** *(main)*, `focal_tversky`, `wce`  
**Selection threshold** (gsi/rf): normalized score `score_threshold=0.5`

Train/val/test = spatial block split. Metrics + artifacts → MLflow; checkpoints → `ml_models/`.

> GPU required. Start with the short smoke run.

In [ ]:
# Register this repo as `crop_mapping_pipeline` regardless of checkout dir name,
# and silence MLflow telemetry.
import os, sys, importlib.util, importlib.machinery
from pathlib import Path

REPO = Path.cwd()
if REPO.name == 'notebooks':
    REPO = REPO.parent
os.environ['MLFLOW_DISABLE_TELEMETRY'] = 'true'

_pkg = 'crop_mapping_pipeline'
if _pkg not in sys.modules:
    _spec = importlib.machinery.ModuleSpec(_pkg, None, is_package=True)
    _mod = importlib.util.module_from_spec(_spec)
    _mod.__path__ = [str(REPO)]
    sys.modules[_pkg] = _mod

from crop_mapping_pipeline import config as C
print('Repo   :', REPO)
print('Classes:', C.NUM_CLASSES, '| crops:', list(C.CDL_CLASS_NAMES.values()))
print('Norms  :', C.__dict__.get('NORM_MODES', ('percentile', 'minmax', 'zscore')))

In [ ]:
from crop_mapping_pipeline.stages.training import train_segmentation as T
MAIN_NORM = 'percentile'          # main normalization
MAIN_LOSS = 'dynamic_balanced'    # DECB-CE, main loss
THRESH    = 0.5                   # gsi/rf normalized-score threshold
print('MLflow:', C.MLFLOW_TRACKING_URI, '| batch', C.BATCH_SIZE, '| max epochs', C.MAX_EPOCHS)
for a, cfg in C.ARCH_CFG.items(): print(f'  {a}: {cfg}')

## 1. Smoke run

GSI / SegFormer, main norm + loss, few epochs — verifies data → split → train → eval.

In [ ]:
T.main(
    exps=['gsi'],
    archs=['segformer'],
    score_threshold=THRESH,
    norm_mode=MAIN_NORM,
    loss=MAIN_LOSS,
    epochs=3,        # smoke; drop to use config MAX_EPOCHS
    skip_ndvi=True,
)

## 2. Full scenario × architecture matrix

All 4 scenarios × 2 architectures, main norm + loss. Uncomment to launch (long).

In [ ]:
# T.main(
#     exps=['single_date', 'mt_ndvi', 'gsi', 'rf'],
#     archs=['deeplabv3plus_cbam', 'segformer'],
#     score_threshold=THRESH,
#     norm_mode=MAIN_NORM,
#     loss=MAIN_LOSS,
# )

## 3. Normalization ablation

Hold scenario/arch/loss fixed; vary normalization ∈ {percentile, minmax, zscore}.

In [ ]:
# for nm in ('percentile', 'minmax', 'zscore'):
#     T.main(exps=['gsi'], archs=['segformer'], score_threshold=THRESH,
#            norm_mode=nm, loss=MAIN_LOSS)

## 4. Loss ablation

Vary loss ∈ {dynamic_balanced (DECB-CE), focal_tversky, wce}.

In [ ]:
# for ls in ('dynamic_balanced', 'focal_tversky', 'wce'):
#     T.main(exps=['gsi'], archs=['segformer'], score_threshold=THRESH,
#            norm_mode=MAIN_NORM, loss=ls)

## 5. Seed-grid stability

Re-runs per seed (re-seeds the spatial split; tags runs `_seed{N}`). Full grid via CLI:

```bash
python stages/train_segmentation.py --exp gsi --arch segformer \
       --score-threshold 0.5 --seed-grid 42 123 456 789
```

In [ ]:
# for s in (42, 123, 456):
#     T.main(exps=['gsi'], archs=['segformer'], score_threshold=THRESH,
#            norm_mode=MAIN_NORM, loss=MAIN_LOSS, seed=s)